# NB14 - Fresh native and AVI batch validation before modularization

## Purpose

This notebook tests whether the fresh July 9, 2026 recordings fit the file and timing assumptions used by NB13.

The fresh data were collected as two export types from the ultrasound machine: native Mindray export folders and AVI recordings. The AVI files were added into each recording folder under `linked_avi/` so the notebook can test paired native/AVI availability.

## Scope

### In scope

- Fresh native recording folder manifest.
- Presence of manually paired `linked_avi/*.avi` files.
- Presence of native `DcmRegionPara.txt`.
- Presence of native `PW_CinePartition0.bin`.
- Fixed-page PW file size check using `1296` bytes per page.
- Basic duration estimate using the current `500 Hz` page-rate assumption.

### Out of scope

- Assuming the `linked_avi/` files were automatically produced inside the native export.
- Package modularization.
- Changing pipeline defaults.
- Clinical validation.
- Native spectrogram recovery.
- Native velocity-envelope recovery.
- Native PSV, EDV, RI, PI, or VTI recovery.

## Interpretation boundary

This notebook tests whether the fresh native exports and manually paired AVI recordings are structurally compatible with the current DopplerLab validation assumptions.

Passing this notebook does not validate clinical measurements, prove native velocity recovery, or prove that native-derived features should become pipeline defaults.

In [16]:
from pathlib import Path
from scipy import signal as sg
from scipy.ndimage import uniform_filter1d
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt, hilbert
import re, cv2, subprocess, tempfile
import pandas as pd
import numpy as np


FRESH_NATIVE_DIR = Path(r"D:\code\DopplerLab\ultrasound_recordings\batch_2026_07_09_native")
PAGE_SIZE_BYTES = 1296
PAGE_RATE_HZ = 500.0

rows = []

for recording_dir in sorted(FRESH_NATIVE_DIR.glob("*SMP")):
    native_dir = recording_dir / "native"
    pw_path = native_dir / "PW_CinePartition0.bin"
    dcm_path = native_dir / "DcmRegionPara.txt"
    linked_avi_paths = sorted((recording_dir / "linked_avi").glob("*.avi"))

    pw_size = pw_path.stat().st_size if pw_path.exists() else np.nan
    n_pages = int(pw_size // PAGE_SIZE_BYTES) if pw_path.exists() else np.nan
    trailing_bytes = int(pw_size % PAGE_SIZE_BYTES) if pw_path.exists() else np.nan

    rows.append(
        {
            "recording_id": recording_dir.name,
            "has_native_dir": native_dir.exists(),
            "has_pw_file": pw_path.exists(),
            "pw_size_bytes": pw_size,
            "pw_pages_1296": n_pages,
            "pw_trailing_bytes": trailing_bytes,
            "native_duration_s": n_pages / PAGE_RATE_HZ if pd.notna(n_pages) else np.nan,
            "has_dcm_region": dcm_path.exists(),
            "dcm_region_size_bytes": dcm_path.stat().st_size if dcm_path.exists() else np.nan,
            "linked_avi_count": len(linked_avi_paths),
            "linked_avi_name": linked_avi_paths[0].name if linked_avi_paths else None,
            "linked_avi_size_bytes": linked_avi_paths[0].stat().st_size if linked_avi_paths else np.nan,
        }
    )

fresh_manifest = pd.DataFrame(rows)

print("Fresh native batch manifest")
print("-" * 88)
print(f"fresh native dir:                 {FRESH_NATIVE_DIR}")
print(f"recording folders:                {len(fresh_manifest)}")
print(f"missing PW files:                 {int((~fresh_manifest['has_pw_file']).sum())}")
print(f"missing DcmRegionPara files:       {int((~fresh_manifest['has_dcm_region']).sum())}")
print(f"recordings without linked AVI:     {int((fresh_manifest['linked_avi_count'] == 0).sum())}")
print(f"PW files with trailing bytes:      {int((fresh_manifest['pw_trailing_bytes'] != 0).sum())}")
print("-" * 88)

display(fresh_manifest)

if len(fresh_manifest) == 0:
    raise ValueError("No fresh native recording folders were found.")

if (~fresh_manifest["has_pw_file"]).any():
    raise ValueError("At least one recording is missing PW_CinePartition0.bin.")

if (~fresh_manifest["has_dcm_region"]).any():
    raise ValueError("At least one recording is missing DcmRegionPara.txt.")

if (fresh_manifest["linked_avi_count"] == 0).any():
    raise ValueError("At least one recording is missing linked AVI.")

if (fresh_manifest["pw_trailing_bytes"] != 0).any():
    raise ValueError("At least one PW file does not divide cleanly into 1296-byte pages.")

Fresh native batch manifest
----------------------------------------------------------------------------------------
fresh native dir:                 D:\code\DopplerLab\ultrasound_recordings\batch_2026_07_09_native
recording folders:                9
missing PW files:                 0
missing DcmRegionPara files:       0
recordings without linked AVI:     0
PW files with trailing bytes:      0
----------------------------------------------------------------------------------------


,recording_id,has_native_dir,has_pw_file,pw_size_bytes,pw_pages_1296,pw_trailing_bytes,native_duration_s,has_dcm_region,dcm_region_size_bytes,linked_avi_count,linked_avi_name,linked_avi_size_bytes
0,202607090207400001SMP,True,True,26151984,20179,0,40.358,True,947,1,202607090207400001SMP.avi,9368272
1,202607090212220002SMP,True,True,20167056,15561,0,31.122,True,947,1,202607090212220002SMP.avi,7803066
2,202607090213000003SMP,True,True,3481056,2686,0,5.372,True,947,1,202607090213000003SMP.avi,1083960
3,202607090216270004SMP,True,True,13693536,10566,0,21.132,True,947,1,202607090216270004SMP.avi,6396524
4,202607090217380005SMP,True,True,17564688,13553,0,27.106,True,947,1,202607090217380005SMP.avi,8613700
5,202607090223110006SMP,True,True,21500640,16590,0,33.180,True,947,1,202607090223110006SMP.avi,9438760
6,202607090226120007SMP,True,True,10213776,7881,0,15.762,True,947,1,202607090226120007SMP.avi,5493160
7,202607090229140008SMP,True,True,23549616,18171,0,36.342,True,947,1,202607090229140008SMP.avi,11061350
8,202607090230590009SMP,True,True,9920880,7655,0,15.310,True,947,1,202607090230590009SMP.avi,4653762


## Fresh DcmRegionPara metadata extraction

### What this tests

This section tests whether each fresh native recording contains a PW `DcmRegionPara` entry with ROI bounds, zero-velocity baseline row, velocity scale, and time scale metadata.

### Why this matters

The NB13 metadata parser should generalize beyond the original June 13 batch before it is modularized. Fresh data provide a first compatibility check.


In [4]:
def parse_dcm_region_pw(native_dir):
    """
    This function reads the PW DcmRegionPara entry from a native Mindray export folder.

    The returned values are metadata-derived ROI and calibration inputs.
    They are not clinical measurements by themselves.
    """
    native_dir = Path(native_dir)
    dcm_region_path = native_dir / "DcmRegionPara.txt"

    if not dcm_region_path.exists():
        raise FileNotFoundError(f"DcmRegionPara.txt was not found: {dcm_region_path}")

    text = dcm_region_path.read_text(errors="replace")
    blocks = re.split(r"DATA_TREE_BEGIN=DcmRegion\d+", text)

    for block in blocks:
        key_values = dict(re.findall(r"(\w+)=([-\d.]+)", block))

        if key_values.get("DataType") == "3" and key_values.get("SpatialFormat") == "3":
            x0 = int(key_values["X0"])
            x1 = int(key_values["X1"])
            y0 = int(key_values["Y0"])
            y1 = int(key_values["Y1"])
            vir_y = int(key_values["VirY"])

            return {
                "x0": x0,
                "x1": x1,
                "y0": y0,
                "y1": y1,
                "vir_y": vir_y,
                "baseline_y_global": y0 + vir_y,
                "baseline_y_roi": vir_y,
                "cm_s_per_px": abs(float(key_values["PhyDeltaY"])),
                "s_per_px": abs(float(key_values["PhyDeltaX"])),
                "data_type": int(key_values["DataType"]),
                "spatial_format": int(key_values["SpatialFormat"]),
            }

    raise ValueError(f"No PW DcmRegion entry found in {dcm_region_path}")

In [5]:
fresh_metadata_rows = []

for recording_id in fresh_manifest["recording_id"]:
    native_dir = FRESH_NATIVE_DIR / recording_id / "native"
    metadata = parse_dcm_region_pw(native_dir)

    fresh_metadata_rows.append(
        {
            "recording_id": recording_id,
            "native_roi": f"{metadata['x0']},{metadata['x1']},{metadata['y0']},{metadata['y1']}",
            "baseline_y_global": metadata["baseline_y_global"],
            "baseline_y_roi": metadata["baseline_y_roi"],
            "cm_s_per_px": metadata["cm_s_per_px"],
            "s_per_px": metadata["s_per_px"],
            "data_type": metadata["data_type"],
            "spatial_format": metadata["spatial_format"],
        }
    )

fresh_metadata = pd.DataFrame(fresh_metadata_rows)

print("Fresh DcmRegionPara metadata extraction")
print("-" * 88)
print(f"recordings parsed:                 {len(fresh_metadata)}")
print(f"unique native ROI values:          {fresh_metadata['native_roi'].nunique()}")
print(f"unique baseline values:            {fresh_metadata['baseline_y_global'].nunique()}")
print(f"unique cm/s/px values:             {fresh_metadata['cm_s_per_px'].nunique()}")
print(f"unique s/px values:                {fresh_metadata['s_per_px'].nunique()}")
print("-" * 88)

display(fresh_metadata)

if len(fresh_metadata) != len(fresh_manifest):
    raise ValueError("Not all fresh recordings produced metadata rows.")

if fresh_metadata[["cm_s_per_px", "s_per_px"]].isna().any().any():
    raise ValueError("At least one fresh metadata row has missing scale values.")

Fresh DcmRegionPara metadata extraction
----------------------------------------------------------------------------------------
recordings parsed:                 9
unique native ROI values:          1
unique baseline values:            1
unique cm/s/px values:             1
unique s/px values:                1
----------------------------------------------------------------------------------------


,recording_id,native_roi,baseline_y_global,baseline_y_roi,cm_s_per_px,s_per_px,data_type,spatial_format
0,202607090207400001SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
1,202607090212220002SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
2,202607090213000003SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
3,202607090216270004SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
4,202607090217380005SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
5,202607090223110006SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
6,202607090226120007SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
7,202607090229140008SMP,"79,659,245,508",376,131,0.176946,0.006,3,3
8,202607090230590009SMP,"79,659,245,508",376,131,0.176946,0.006,3,3


## Fresh native PW page structure and counter integrity

### What this tests

This section tests whether fresh `PW_CinePartition0.bin` files behave as fixed-size native PW page streams with a reproducible counter-like timing field.

### Why this matters

Native timing/QC helpers depend on stable page structure. This check tests whether the `1296` byte page size, `500 Hz` page-rate assumption, and 20 Hz counter pattern generalize to the fresh July 9 batch.


In [6]:
PAGE_SIZE_BYTES = 1296
PAGE_RATE_HZ = 500.0
COUNTER_BYTE_OFFSET = 1248


def load_native_pw_pages(pw_path, page_size_bytes=PAGE_SIZE_BYTES):
    """
    This function reads PW_CinePartition0.bin as fixed-size native pages.
    """
    pw_path = Path(pw_path)

    if not pw_path.exists():
        raise FileNotFoundError(f"Native PW file was not found: {pw_path}")

    raw_bytes = np.fromfile(pw_path, dtype=np.uint8)
    n_pages = raw_bytes.size // page_size_bytes
    trailing_bytes = raw_bytes.size - (n_pages * page_size_bytes)

    if n_pages == 0:
        raise ValueError(f"No complete native PW pages found in {pw_path}")

    pages = raw_bytes[: n_pages * page_size_bytes].reshape(n_pages, page_size_bytes)

    return {
        "pages": pages,
        "n_pages": int(n_pages),
        "file_size_bytes": int(raw_bytes.size),
        "trailing_bytes": int(trailing_bytes),
    }


def summarize_native_counter(
    pages,
    counter_byte_offset=COUNTER_BYTE_OFFSET,
    page_rate_hz=PAGE_RATE_HZ,
):
    """
    This function summarizes the counter-like uint16 field in native PW pages.
    """
    counter = (
        pages[:, counter_byte_offset].astype(np.uint16)
        | (pages[:, counter_byte_offset + 1].astype(np.uint16) << 8)
    ).astype(np.int64)

    counter_diff = np.diff(counter)
    positive_increment_locations = np.where(counter_diff > 0)[0]

    if positive_increment_locations.size > 2:
        counter_group_pages = float(np.median(np.diff(positive_increment_locations)))
    else:
        counter_group_pages = np.nan

    counter_group_hz = (
        float(page_rate_hz / counter_group_pages)
        if np.isfinite(counter_group_pages) and counter_group_pages > 0
        else np.nan
    )

    return {
        "counter_unique": int(np.unique(counter).size),
        "counter_group_pages": counter_group_pages,
        "counter_group_hz": counter_group_hz,
        "counter_resets": int((counter_diff < -100).sum()),
        "counter_monotone_fraction": float((counter_diff >= 0).mean()),
    }

In [7]:
fresh_counter_rows = []

for recording_id in fresh_manifest["recording_id"]:
    pw_path = FRESH_NATIVE_DIR / recording_id / "native" / "PW_CinePartition0.bin"

    page_info = load_native_pw_pages(pw_path)
    counter_summary = summarize_native_counter(page_info["pages"])

    native_duration_s = page_info["n_pages"] / PAGE_RATE_HZ
    duration_status = "short_duration_caution" if native_duration_s < 10.0 else "duration_ok"

    fresh_counter_rows.append(
        {
            "recording_id": recording_id,
            "file_size_bytes": page_info["file_size_bytes"],
            "n_pages": page_info["n_pages"],
            "trailing_bytes": page_info["trailing_bytes"],
            "native_duration_s": native_duration_s,
            "duration_status": duration_status,
            "counter_unique": counter_summary["counter_unique"],
            "counter_group_pages": counter_summary["counter_group_pages"],
            "counter_group_hz": counter_summary["counter_group_hz"],
            "counter_resets": counter_summary["counter_resets"],
            "counter_monotone_fraction": counter_summary["counter_monotone_fraction"],
        }
    )

fresh_counter_summary = pd.DataFrame(fresh_counter_rows)

print("Fresh native PW page structure and counter integrity")
print("-" * 88)
print(f"fresh PW files checked:            {len(fresh_counter_summary)}")
print(f"page size bytes:                   {PAGE_SIZE_BYTES}")
print(f"page rate Hz assumption:           {PAGE_RATE_HZ:.1f}")
print(f"files with trailing bytes:         {int((fresh_counter_summary['trailing_bytes'] != 0).sum())}")
print(f"median counter group pages:        {fresh_counter_summary['counter_group_pages'].median():.1f}")
print(f"median counter group Hz:           {fresh_counter_summary['counter_group_hz'].median():.1f}")
print(f"total counter resets:              {int(fresh_counter_summary['counter_resets'].sum())}")
print(f"short duration cautions:           {int((fresh_counter_summary['duration_status'] == 'short_duration_caution').sum())}")
print("-" * 88)

display(fresh_counter_summary)

if (fresh_counter_summary["trailing_bytes"] != 0).any():
    raise ValueError("At least one fresh PW file does not divide cleanly into 1296-byte pages.")

if not np.allclose(fresh_counter_summary["counter_group_pages"], 25.0, equal_nan=False):
    raise ValueError("At least one fresh PW counter group interval is not 25 pages.")

if fresh_counter_summary["counter_resets"].sum() != 0:
    raise ValueError("At least one fresh PW counter reset was detected.")

if (fresh_counter_summary["counter_monotone_fraction"] < 0.99).any():
    raise ValueError("At least one fresh PW counter is not mostly monotone.")

Fresh native PW page structure and counter integrity
----------------------------------------------------------------------------------------
fresh PW files checked:            9
page size bytes:                   1296
page rate Hz assumption:           500.0
files with trailing bytes:         0
median counter group pages:        25.0
median counter group Hz:           20.0
total counter resets:              0
short duration cautions:           1
----------------------------------------------------------------------------------------


,recording_id,file_size_bytes,n_pages,trailing_bytes,native_duration_s,duration_status,counter_unique,counter_group_pages,counter_group_hz,counter_resets,counter_monotone_fraction
0,202607090207400001SMP,26151984,20179,0,40.358,duration_ok,804,25.0,20.0,0,1.0
1,202607090212220002SMP,20167056,15561,0,31.122,duration_ok,620,25.0,20.0,0,1.0
2,202607090213000003SMP,3481056,2686,0,5.372,short_duration_caution,107,25.0,20.0,0,1.0
3,202607090216270004SMP,13693536,10566,0,21.132,duration_ok,421,25.0,20.0,0,1.0
4,202607090217380005SMP,17564688,13553,0,27.106,duration_ok,540,25.0,20.0,0,1.0
5,202607090223110006SMP,21500640,16590,0,33.180,duration_ok,661,25.0,20.0,0,1.0
6,202607090226120007SMP,10213776,7881,0,15.762,duration_ok,314,25.0,20.0,0,1.0
7,202607090229140008SMP,23549616,18171,0,36.342,duration_ok,724,25.0,20.0,0,1.0
8,202607090230590009SMP,9920880,7655,0,15.310,duration_ok,305,25.0,20.0,0,1.0


## Fresh native hp_activity timing/QC sidecar

### What this tests

This section tests whether experimental native `hp_activity` timing/QC features can be reproduced from fresh raw `PW_CinePartition0.bin` files.

### Why this matters

The native timing/QC sidecar should generalize beyond the original NB13 batch before it is modularized. The short recording is retained as a stress test and should receive a duration caution rather than causing the batch to fail.


In [9]:
HP_ENV_RATE_HZ = 100.0


def load_native_pw_record_field_b(pw_path):
    """
    This function loads the compact dynamic record field used for experimental
    native timing/QC sidecar extraction.
    """
    page_info = load_native_pw_pages(pw_path)
    pages = page_info["pages"]
    n_pages = page_info["n_pages"]

    records = pages[:, 16:144].reshape(n_pages, 16, 8)

    field_b = (
        np.ascontiguousarray(records[:, :14, 4:8])
        .reshape(-1, 4)
        .view("<f4")
        .reshape(n_pages, 14)
    )

    field_b = np.nan_to_num(field_b.astype(np.float64))

    return field_b, page_info


def compute_hp_activity(field_b, smoothing_window_pages=51):
    """
    This function computes an arbitrary-unit high-pass activity trace from
    compact native PW page records.
    """
    smoothed = uniform_filter1d(
        field_b,
        size=smoothing_window_pages,
        axis=0,
        mode="nearest",
    )

    return np.abs(field_b - smoothed).mean(axis=1)


def compute_activity_envelope(
    activity,
    page_rate_hz=PAGE_RATE_HZ,
    envelope_rate_hz=HP_ENV_RATE_HZ,
    band_hz=(0.5, 12.0),
):
    """
    This function computes a low-rate envelope from native hp_activity.
    """
    activity = np.asarray(activity, dtype=float)

    if activity.ndim != 1:
        raise ValueError("activity must be a 1D array.")

    if not np.isfinite(activity).all():
        raise ValueError("activity contains non-finite values.")

    centered = activity - activity.mean()

    sos = sg.butter(
        N=4,
        Wn=[band_hz[0] / (page_rate_hz / 2.0), band_hz[1] / (page_rate_hz / 2.0)],
        btype="band",
        output="sos",
    )

    filtered = sg.sosfiltfilt(sos, centered)
    envelope = np.abs(sg.hilbert(filtered))

    downsample_factor = int(round(page_rate_hz / envelope_rate_hz))

    return envelope[::downsample_factor]


def estimate_periodic_hr_bpm(envelope, envelope_rate_hz=HP_ENV_RATE_HZ, low_bpm=40, high_bpm=200):
    """
    This function estimates a dominant HR-like periodicity from an envelope
    autocorrelation.
    """
    envelope = np.asarray(envelope, dtype=float)

    if envelope.ndim != 1:
        raise ValueError("envelope must be a 1D array.")

    centered = envelope - envelope.mean()

    if centered.std() == 0:
        return {
            "autocorr_peak": 0.0,
            "hr_bpm": np.nan,
        }

    n_samples = len(centered)
    fft_values = np.fft.rfft(centered, n=2 * n_samples)
    autocorr = np.fft.irfft(fft_values * np.conj(fft_values))[:n_samples]
    autocorr = autocorr / autocorr[0]

    min_lag = int(envelope_rate_hz * 60.0 / high_bpm)
    max_lag = min(int(envelope_rate_hz * 60.0 / low_bpm), n_samples - 1)

    if max_lag <= min_lag:
        return {
            "autocorr_peak": np.nan,
            "hr_bpm": np.nan,
        }

    local_peak_index = int(np.argmax(autocorr[min_lag:max_lag]))
    lag_samples = min_lag + local_peak_index
    hr_bpm = 60.0 / (lag_samples / envelope_rate_hz)

    return {
        "autocorr_peak": float(autocorr[lag_samples]),
        "hr_bpm": float(hr_bpm),
    }


def summarize_native_hp_activity_sidecar(pw_path):
    """
    This function derives experimental native hp_activity timing/QC features
    from one raw PW_CinePartition0.bin file.
    """
    field_b, page_info = load_native_pw_record_field_b(pw_path)
    hp_activity = compute_hp_activity(field_b)
    envelope = compute_activity_envelope(hp_activity)
    periodicity = estimate_periodic_hr_bpm(envelope)

    native_duration_s = page_info["n_pages"] / PAGE_RATE_HZ
    duration_status = "short_duration_caution" if native_duration_s < 10.0 else "duration_ok"

    window_length = len(envelope) // 5
    window_hr_values = []

    if window_length > HP_ENV_RATE_HZ * 4:
        for window_index in range(5):
            start = window_index * window_length
            stop = (window_index + 1) * window_length
            window_result = estimate_periodic_hr_bpm(envelope[start:stop])
            window_hr_values.append(window_result["hr_bpm"])

    window_hr_values = np.asarray(window_hr_values, dtype=float)
    finite_window_hr = window_hr_values[np.isfinite(window_hr_values)]

    if len(finite_window_hr) > 2:
        window_hr_iqr_bpm = float(
            np.percentile(finite_window_hr, 75)
            - np.percentile(finite_window_hr, 25)
        )
    else:
        window_hr_iqr_bpm = np.nan

    def subset_hr(column_indices):
        subset_activity = compute_hp_activity(field_b[:, column_indices])
        subset_envelope = compute_activity_envelope(subset_activity)
        return estimate_periodic_hr_bpm(subset_envelope)["hr_bpm"]

    first_half_hr = subset_hr(list(range(0, 7)))
    second_half_hr = subset_hr(list(range(7, 14)))
    even_hr = subset_hr(list(range(0, 14, 2)))
    odd_hr = subset_hr(list(range(1, 14, 2)))

    subset_disagreement_bpm = float(
        np.mean(
            [
                abs(first_half_hr - second_half_hr),
                abs(even_hr - odd_hr),
            ]
        )
    )

    native_specific_candidate = bool(
        duration_status == "duration_ok"
        and np.isfinite(window_hr_iqr_bpm)
        and window_hr_iqr_bpm < 8.0
        and subset_disagreement_bpm < 6.0
    )

    return {
        "n_pages": page_info["n_pages"],
        "native_duration_s": native_duration_s,
        "duration_status": duration_status,
        "hp_activity_finite": bool(np.isfinite(hp_activity).all()),
        "hp_activity_min": float(np.min(hp_activity)),
        "hp_activity_max": float(np.max(hp_activity)),
        "native_hr_bpm": round(periodicity["hr_bpm"], 1),
        "autocorr_peak": round(periodicity["autocorr_peak"], 3),
        "window_hr_iqr_bpm": round(window_hr_iqr_bpm, 1) if np.isfinite(window_hr_iqr_bpm) else np.nan,
        "subset_disagreement_bpm": round(subset_disagreement_bpm, 1),
        "native_specific_candidate": native_specific_candidate,
    }

In [10]:
fresh_sidecar_rows = []

for recording_id in fresh_manifest["recording_id"]:
    pw_path = FRESH_NATIVE_DIR / recording_id / "native" / "PW_CinePartition0.bin"
    sidecar = summarize_native_hp_activity_sidecar(pw_path)

    fresh_sidecar_rows.append(
        {
            "recording_id": recording_id,
            "native_duration_s": sidecar["native_duration_s"],
            "duration_status": sidecar["duration_status"],
            "native_hr_bpm": sidecar["native_hr_bpm"],
            "autocorr_peak": sidecar["autocorr_peak"],
            "window_hr_iqr_bpm": sidecar["window_hr_iqr_bpm"],
            "subset_disagreement_bpm": sidecar["subset_disagreement_bpm"],
            "native_specific_candidate": sidecar["native_specific_candidate"],
            "hp_activity_finite": sidecar["hp_activity_finite"],
            "hp_activity_min": sidecar["hp_activity_min"],
            "hp_activity_max": sidecar["hp_activity_max"],
        }
    )

fresh_hp_sidecar = pd.DataFrame(fresh_sidecar_rows)

print("Fresh native hp_activity timing/QC sidecar")
print("-" * 88)
print(f"fresh sidecars reproduced:         {len(fresh_hp_sidecar)}")
print(f"hp_activity finite for all:        {bool(fresh_hp_sidecar['hp_activity_finite'].all())}")
print(f"duration cautions:                 {int((fresh_hp_sidecar['duration_status'] == 'short_duration_caution').sum())}")
print(f"native-specific candidates:        {int(fresh_hp_sidecar['native_specific_candidate'].sum())}")
print("-" * 88)

display(fresh_hp_sidecar)

if len(fresh_hp_sidecar) != len(fresh_manifest):
    raise ValueError("Not all fresh recordings produced sidecar rows.")

if not fresh_hp_sidecar["hp_activity_finite"].all():
    raise ValueError("At least one fresh hp_activity trace contained non-finite values.")

Fresh native hp_activity timing/QC sidecar
----------------------------------------------------------------------------------------
fresh sidecars reproduced:         9
hp_activity finite for all:        True
duration cautions:                 1
native-specific candidates:        2
----------------------------------------------------------------------------------------


,recording_id,native_duration_s,duration_status,native_hr_bpm,autocorr_peak,window_hr_iqr_bpm,subset_disagreement_bpm,native_specific_candidate,hp_activity_finite,hp_activity_min,hp_activity_max
0,202607090207400001SMP,40.358,duration_ok,78.9,0.444,2.0,0.5,True,True,0.002030,5.056956
1,202607090212220002SMP,31.122,duration_ok,200.0,0.168,7.1,0.0,True,True,0.003750,4.070743
2,202607090213000003SMP,5.372,short_duration_caution,200.0,0.146,NaN,67.7,False,True,0.001915,4.485234
3,202607090216270004SMP,21.132,duration_ok,200.0,0.154,149.2,16.2,False,True,0.000986,4.727884
4,202607090217380005SMP,27.106,duration_ok,93.8,0.261,45.1,23.3,False,True,0.003730,5.035836
5,202607090223110006SMP,33.180,duration_ok,122.4,0.064,95.6,43.3,False,True,0.007667,3.603788
6,202607090226120007SMP,15.762,duration_ok,56.6,0.460,NaN,56.9,False,True,0.004588,5.814171
7,202607090229140008SMP,36.342,duration_ok,153.8,0.188,56.8,25.0,False,True,0.001076,3.529151
8,202607090230590009SMP,15.310,duration_ok,76.9,0.140,NaN,48.7,False,True,0.003899,5.447463


### Interpretation - fresh native hp_activity timing/QC sidecar

**Result classification:** fresh native `hp_activity` sidecar reproduced, with quality stratification.

The fresh batch intentionally includes short and poor-quality recordings to test pipeline robustness. The sidecar extraction produced finite outputs for all recordings and labeled the short recording with `short_duration_caution` rather than failing the batch.

Only two recordings met the current native-specific candidate rule. Several recordings produced unstable window or subset metrics, and some HR-like estimates reached the upper search boundary. These results are useful QC evidence, not failures.

The result supports modularizing native page loading, counter checks, and `hp_activity` sidecar extraction as experimental additive QC helpers.

The result does not support treating `hp_activity` as a native velocity envelope, clinical HR, decoded spectrogram, or default acceptance criterion.

## Fresh manually paired AVI readability

### What this tests

This section tests whether each manually paired `linked_avi/*.avi` file can be opened and whether basic video metadata can be read.

### Why this matters

The July 9 data were collected as separate native exports and AVI recordings. Before any native/AVI comparison, the paired AVI files must be readable and have usable frame timing metadata.



In [11]:
def read_basic_avi_metadata(avi_path):
    """
    This function reads basic video metadata from an AVI file using OpenCV.
    """
    avi_path = Path(avi_path)

    if not avi_path.exists():
        return {
            "avi_exists": False,
            "video_opened": False,
            "fps": np.nan,
            "frame_count": np.nan,
            "duration_s": np.nan,
            "width": np.nan,
            "height": np.nan,
            "first_frame_read": False,
        }

    cap = cv2.VideoCapture(str(avi_path))
    opened = bool(cap.isOpened())

    if not opened:
        cap.release()
        return {
            "avi_exists": True,
            "video_opened": False,
            "fps": np.nan,
            "frame_count": np.nan,
            "duration_s": np.nan,
            "width": np.nan,
            "height": np.nan,
            "first_frame_read": False,
        }

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    ok, _ = cap.read()
    cap.release()

    duration_s = frame_count / fps if fps > 0 else np.nan

    return {
        "avi_exists": True,
        "video_opened": opened,
        "fps": fps,
        "frame_count": frame_count,
        "duration_s": duration_s,
        "width": width,
        "height": height,
        "first_frame_read": bool(ok),
    }

In [13]:
fresh_avi_rows = []

for _, manifest_row in fresh_manifest.iterrows():
    recording_id = manifest_row["recording_id"]
    linked_avi_path = FRESH_NATIVE_DIR / recording_id / "linked_avi" / manifest_row["linked_avi_name"]

    avi_metadata = read_basic_avi_metadata(linked_avi_path)

    native_duration_s = float(
        fresh_counter_summary.loc[
            fresh_counter_summary["recording_id"] == recording_id,
            "native_duration_s",
        ].iloc[0]
    )

    fresh_avi_rows.append(
        {
            "recording_id": recording_id,
            "linked_avi_name": manifest_row["linked_avi_name"],
            "linked_avi_size_bytes": manifest_row["linked_avi_size_bytes"],
            "avi_exists": avi_metadata["avi_exists"],
            "video_opened": avi_metadata["video_opened"],
            "first_frame_read": avi_metadata["first_frame_read"],
            "fps": avi_metadata["fps"],
            "frame_count": avi_metadata["frame_count"],
            "avi_duration_s": avi_metadata["duration_s"],
            "native_duration_s": native_duration_s,
            "avi_minus_native_duration_s": (
                avi_metadata["duration_s"] - native_duration_s
                if pd.notna(avi_metadata["duration_s"])
                else np.nan
            ),
            "width": avi_metadata["width"],
            "height": avi_metadata["height"],
        }
    )

fresh_avi_metadata = pd.DataFrame(fresh_avi_rows)

print("Fresh manually paired AVI readability")
print("-" * 88)
print(f"AVI files checked:                 {len(fresh_avi_metadata)}")
print(f"AVI files opened:                  {int(fresh_avi_metadata['video_opened'].sum())}")
print(f"first frames readable:             {int(fresh_avi_metadata['first_frame_read'].sum())}")
print(f"unique FPS values:                 {fresh_avi_metadata['fps'].dropna().unique()}")
print(f"unique frame sizes:                {fresh_avi_metadata[['width', 'height']].drop_duplicates().shape[0]}")
print("-" * 88)

display(fresh_avi_metadata)

if not fresh_avi_metadata["video_opened"].all():
    raise ValueError("At least one linked AVI could not be opened.")

if not fresh_avi_metadata["first_frame_read"].all():
    raise ValueError("At least one linked AVI first frame could not be read.")

Fresh manually paired AVI readability
----------------------------------------------------------------------------------------
AVI files checked:                 9
AVI files opened:                  9
first frames readable:             9
unique FPS values:                 [30.]
unique frame sizes:                1
----------------------------------------------------------------------------------------


,recording_id,linked_avi_name,linked_avi_size_bytes,avi_exists,video_opened,first_frame_read,fps,frame_count,avi_duration_s,native_duration_s,avi_minus_native_duration_s,width,height
0,202607090207400001SMP,202607090207400001SMP.avi,9368272,True,True,True,30.0,1212,40.400000,40.358,0.042000,720,540
1,202607090212220002SMP,202607090212220002SMP.avi,7803066,True,True,True,30.0,935,31.166667,31.122,0.044667,720,540
2,202607090213000003SMP,202607090213000003SMP.avi,1083960,True,True,True,30.0,163,5.433333,5.372,0.061333,720,540
3,202607090216270004SMP,202607090216270004SMP.avi,6396524,True,True,True,30.0,635,21.166667,21.132,0.034667,720,540
4,202607090217380005SMP,202607090217380005SMP.avi,8613700,True,True,True,30.0,815,27.166667,27.106,0.060667,720,540
5,202607090223110006SMP,202607090223110006SMP.avi,9438760,True,True,True,30.0,997,33.233333,33.180,0.053333,720,540
6,202607090226120007SMP,202607090226120007SMP.avi,5493160,True,True,True,30.0,474,15.800000,15.762,0.038000,720,540
7,202607090229140008SMP,202607090229140008SMP.avi,11061350,True,True,True,30.0,1092,36.400000,36.342,0.058000,720,540
8,202607090230590009SMP,202607090230590009SMP.avi,4653762,True,True,True,30.0,461,15.366667,15.310,0.056667,720,540


## Fresh AVI audio extraction and clipping QC

### What this tests

This section tests whether audio can be extracted from each manually paired AVI and whether a simple clipping QC metric can be computed.

### Why this matters

Audio-derived HR comparison requires usable audio. Clipping QC helps identify recordings where audio timing estimates may be less reliable.


In [14]:
def audio_clipping_metrics(wav_path, near_peak_threshold=0.98, clip_fraction_threshold=0.001):
    """
    This function computes a candidate audio clipping metric from a WAV file.
    """
    wav_path = Path(wav_path)

    if not wav_path.exists():
        return {
            "audio_clip_fraction": np.nan,
            "audio_peak": np.nan,
            "audio_possible_clipping_candidate": False,
            "audio_sample_rate_hz": np.nan,
            "audio_duration_s": np.nan,
        }

    sample_rate_hz, audio = wavfile.read(wav_path)
    audio = audio.astype(float)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if len(audio) == 0:
        return {
            "audio_clip_fraction": np.nan,
            "audio_peak": np.nan,
            "audio_possible_clipping_candidate": False,
            "audio_sample_rate_hz": sample_rate_hz,
            "audio_duration_s": 0.0,
        }

    peak = float(np.max(np.abs(audio))) + 1e-9
    clip_fraction = float((np.abs(audio) >= near_peak_threshold * peak).mean())
    touches_int16_rail = peak >= 32767

    candidate_clip = bool(
        clip_fraction > clip_fraction_threshold
        and touches_int16_rail
    )

    return {
        "audio_clip_fraction": round(clip_fraction, 5),
        "audio_peak": int(round(peak)),
        "audio_possible_clipping_candidate": candidate_clip,
        "audio_sample_rate_hz": int(sample_rate_hz),
        "audio_duration_s": len(audio) / float(sample_rate_hz),
    }

In [17]:
fresh_audio_rows = []

for _, avi_row in fresh_avi_metadata.iterrows():
    recording_id = avi_row["recording_id"]
    avi_path = FRESH_NATIVE_DIR / recording_id / "linked_avi" / avi_row["linked_avi_name"]
    wav_path = Path(tempfile.gettempdir()) / f"nb14_fresh_audio_{recording_id}.wav"

    ffmpeg_result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            str(avi_path),
            "-ac",
            "1",
            str(wav_path),
        ],
        capture_output=True,
        text=True,
    )

    if ffmpeg_result.returncode == 0 and wav_path.exists():
        clipping = audio_clipping_metrics(wav_path)
        extraction_status = "audio_extracted"
    else:
        clipping = {
            "audio_clip_fraction": np.nan,
            "audio_peak": np.nan,
            "audio_possible_clipping_candidate": False,
            "audio_sample_rate_hz": np.nan,
            "audio_duration_s": np.nan,
        }
        extraction_status = "audio_extract_failed"

    fresh_audio_rows.append(
        {
            "recording_id": recording_id,
            "audio_extraction_status": extraction_status,
            "audio_sample_rate_hz": clipping["audio_sample_rate_hz"],
            "audio_duration_s": clipping["audio_duration_s"],
            "avi_duration_s": avi_row["avi_duration_s"],
            "audio_minus_avi_duration_s": (
                clipping["audio_duration_s"] - avi_row["avi_duration_s"]
                if pd.notna(clipping["audio_duration_s"])
                else np.nan
            ),
            "audio_clip_fraction": clipping["audio_clip_fraction"],
            "audio_peak": clipping["audio_peak"],
            "audio_possible_clipping_candidate": clipping["audio_possible_clipping_candidate"],
        }
    )

fresh_audio_qc = pd.DataFrame(fresh_audio_rows)

print("Fresh AVI audio extraction and clipping QC")
print("-" * 88)
print(f"recordings checked:                {len(fresh_audio_qc)}")
print(f"audio extracted:                   {int((fresh_audio_qc['audio_extraction_status'] == 'audio_extracted').sum())}")
print(f"audio extraction failures:         {int((fresh_audio_qc['audio_extraction_status'] != 'audio_extracted').sum())}")
print(f"candidate clipping positives:      {int(fresh_audio_qc['audio_possible_clipping_candidate'].sum())}")
print(f"unique audio sample rates:         {fresh_audio_qc['audio_sample_rate_hz'].dropna().unique()}")
print("-" * 88)

display(fresh_audio_qc)

if (fresh_audio_qc["audio_extraction_status"] != "audio_extracted").any():
    raise RuntimeError("At least one fresh AVI audio extraction failed.")

Fresh AVI audio extraction and clipping QC
----------------------------------------------------------------------------------------
recordings checked:                9
audio extracted:                   9
audio extraction failures:         0
candidate clipping positives:      0
unique audio sample rates:         [96000]
----------------------------------------------------------------------------------------


,recording_id,audio_extraction_status,audio_sample_rate_hz,audio_duration_s,avi_duration_s,audio_minus_avi_duration_s,audio_clip_fraction,audio_peak,audio_possible_clipping_candidate
0,202607090207400001SMP,audio_extracted,96000,40.373333,40.400000,-0.026667,0.00001,17090,False
1,202607090212220002SMP,audio_extracted,96000,31.136000,31.166667,-0.030667,0.00001,5936,False
2,202607090213000003SMP,audio_extracted,96000,5.386667,5.433333,-0.046667,0.00050,17352,False
3,202607090216270004SMP,audio_extracted,96000,21.141333,21.166667,-0.025333,0.00001,8026,False
4,202607090217380005SMP,audio_extracted,96000,27.114667,27.166667,-0.052000,0.00001,5621,False
5,202607090223110006SMP,audio_extracted,96000,33.194667,33.233333,-0.038667,0.00001,5178,False
6,202607090226120007SMP,audio_extracted,96000,15.776000,15.800000,-0.024000,0.00002,4371,False
7,202607090229140008SMP,audio_extracted,96000,36.352000,36.400000,-0.048000,0.00001,4722,False
8,202607090230590009SMP,audio_extracted,96000,15.328000,15.366667,-0.038667,0.00010,17504,False


## Fresh audio/native HR timing comparison

### What this tests

This section tests whether rough audio-derived HR estimates and native `hp_activity` HR estimates agree within 5 bpm for the fresh paired recordings.

### Why this matters

Audio and native `hp_activity` are independent timing signals. Agreement can support recording-level QC, while disagreement can flag recordings for review.


In [18]:
def estimate_audio_hr_bpm_from_wav(
    wav_path,
    low_bpm=40,
    high_bpm=200,
    envelope_rate_hz=100.0,
):
    """
    This function estimates a rough HR-like periodicity from extracted Doppler audio.

    The estimate is for timing/QC comparison only and is not a clinical HR measurement.
    """
    sample_rate_hz, audio = wavfile.read(wav_path)
    audio = audio.astype(float)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if len(audio) == 0:
        return {
            "audio_hr_bpm": np.nan,
            "audio_autocorr_peak": np.nan,
        }

    audio = audio - np.mean(audio)

    if np.std(audio) == 0:
        return {
            "audio_hr_bpm": np.nan,
            "audio_autocorr_peak": 0.0,
        }

    nyquist = sample_rate_hz / 2.0
    sos = butter(
        N=4,
        Wn=[300.0 / nyquist, 8000.0 / nyquist],
        btype="band",
        output="sos",
    )

    filtered = sosfiltfilt(sos, audio)
    envelope = np.abs(hilbert(filtered))

    downsample_factor = max(1, int(round(sample_rate_hz / envelope_rate_hz)))
    envelope_low_rate = envelope[::downsample_factor]

    return estimate_periodic_hr_bpm(
        envelope_low_rate,
        envelope_rate_hz=envelope_rate_hz,
        low_bpm=low_bpm,
        high_bpm=high_bpm,
    )


def classify_hr_agreement(abs_diff_bpm):
    """
    This function classifies agreement between two HR-like timing estimates.
    """
    if pd.isna(abs_diff_bpm):
        return "not_assessed"

    if abs_diff_bpm <= 5.0:
        return "agree_5bpm"

    return "disagree_gt_5bpm"

In [20]:
fresh_hr_rows = []

for _, audio_row in fresh_audio_qc.iterrows():
    recording_id = audio_row["recording_id"]
    wav_path = Path(tempfile.gettempdir()) / f"nb14_fresh_audio_{recording_id}.wav"

    if audio_row["audio_extraction_status"] == "audio_extracted" and wav_path.exists():
        audio_hr = estimate_audio_hr_bpm_from_wav(wav_path)
    else:
        audio_hr = {
            "hr_bpm": np.nan,
            "autocorr_peak": np.nan,
        }

    native_row = fresh_hp_sidecar[
        fresh_hp_sidecar["recording_id"] == recording_id
    ].iloc[0]

    audio_hr_bpm = audio_hr.get("hr_bpm", np.nan)
    audio_autocorr_peak = audio_hr.get("autocorr_peak", np.nan)
    native_hr_bpm = float(native_row["native_hr_bpm"])

    abs_diff_bpm = (
        abs(audio_hr_bpm - native_hr_bpm)
        if pd.notna(audio_hr_bpm) and pd.notna(native_hr_bpm)
        else np.nan
    )

    fresh_hr_rows.append(
        {
            "recording_id": recording_id,
            "duration_status": native_row["duration_status"],
            "native_specific_candidate": bool(native_row["native_specific_candidate"]),
            "audio_hr_bpm": round(float(audio_hr_bpm), 1) if pd.notna(audio_hr_bpm) else np.nan,
            "audio_autocorr_peak": round(float(audio_autocorr_peak), 3) if pd.notna(audio_autocorr_peak) else np.nan,
            "native_hr_bpm": round(native_hr_bpm, 1),
            "native_autocorr_peak": native_row["autocorr_peak"],
            "abs_diff_bpm": round(float(abs_diff_bpm), 1) if pd.notna(abs_diff_bpm) else np.nan,
            "hr_agreement_status": classify_hr_agreement(abs_diff_bpm),
        }
    )

fresh_audio_native_hr = pd.DataFrame(fresh_hr_rows)

print("Fresh audio/native HR timing comparison")
print("-" * 88)
print(f"recordings compared:               {len(fresh_audio_native_hr)}")
print("agreement status counts:")
print(fresh_audio_native_hr["hr_agreement_status"].value_counts(dropna=False).to_dict())
print("-" * 88)

display(fresh_audio_native_hr)

if len(fresh_audio_native_hr) != len(fresh_manifest):
    raise ValueError("Not all fresh recordings produced HR comparison rows.")


Fresh audio/native HR timing comparison
----------------------------------------------------------------------------------------
recordings compared:               9
agreement status counts:
{'disagree_gt_5bpm': 6, 'agree_5bpm': 3}
----------------------------------------------------------------------------------------


,recording_id,duration_status,native_specific_candidate,audio_hr_bpm,audio_autocorr_peak,native_hr_bpm,native_autocorr_peak,abs_diff_bpm,hr_agreement_status
0,202607090207400001SMP,duration_ok,True,78.9,0.157,78.9,0.444,0.0,agree_5bpm
1,202607090212220002SMP,duration_ok,True,95.2,0.707,200.0,0.168,104.8,disagree_gt_5bpm
2,202607090213000003SMP,short_duration_caution,False,101.7,0.213,200.0,0.146,98.3,disagree_gt_5bpm
3,202607090216270004SMP,duration_ok,False,92.3,0.627,200.0,0.154,107.7,disagree_gt_5bpm
4,202607090217380005SMP,duration_ok,False,93.8,0.199,93.8,0.261,0.0,agree_5bpm
5,202607090223110006SMP,duration_ok,False,115.4,0.651,122.4,0.064,7.0,disagree_gt_5bpm
6,202607090226120007SMP,duration_ok,False,111.1,0.430,56.6,0.460,54.5,disagree_gt_5bpm
7,202607090229140008SMP,duration_ok,False,78.9,0.675,153.8,0.188,74.9,disagree_gt_5bpm
8,202607090230590009SMP,duration_ok,False,75.0,0.502,76.9,0.140,1.9,agree_5bpm


## Fresh timing QC disagreement review

### What this tests

This section tests whether simple caution labels explain fresh audio/native HR agreement and disagreement patterns.

### Why this matters

Fresh data exposed a useful failure mode: some native HR-like estimates hit the upper search boundary or have weak autocorrelation support. A modularized QC helper should preserve these caution labels rather than returning only a single acceptance Boolean.


In [21]:
def classify_native_hr_qc_status(row):
    """
    This function assigns a cautious native timing/QC status from duration,
    HR boundary behavior, autocorrelation strength, and stability metrics.
    """
    cautions = []

    if row["duration_status"] == "short_duration_caution":
        cautions.append("short_duration")

    if pd.notna(row["native_hr_bpm"]) and row["native_hr_bpm"] >= 195.0:
        cautions.append("native_hr_upper_boundary")

    if pd.notna(row["native_autocorr_peak"]) and row["native_autocorr_peak"] < 0.20:
        cautions.append("weak_native_autocorr")

    if pd.notna(row.get("window_hr_iqr_bpm", np.nan)) and row["window_hr_iqr_bpm"] >= 8.0:
        cautions.append("unstable_window_hr")

    if pd.notna(row.get("subset_disagreement_bpm", np.nan)) and row["subset_disagreement_bpm"] >= 6.0:
        cautions.append("subset_disagreement")

    if len(cautions) == 0:
        return "native_timing_supported"

    return "caution_" + ";".join(cautions)


fresh_timing_review = fresh_audio_native_hr.merge(
    fresh_hp_sidecar[
        [
            "recording_id",
            "window_hr_iqr_bpm",
            "subset_disagreement_bpm",
        ]
    ],
    on="recording_id",
    how="left",
)

fresh_timing_review["native_timing_qc_status"] = fresh_timing_review.apply(
    classify_native_hr_qc_status,
    axis=1,
)

fresh_timing_review["audio_native_agreement_supported"] = (
    fresh_timing_review["hr_agreement_status"] == "agree_5bpm"
)

print("Fresh timing QC disagreement review")
print("-" * 88)
print("HR agreement counts:")
print(fresh_timing_review["hr_agreement_status"].value_counts(dropna=False).to_dict())
print("Native timing QC status counts:")
print(fresh_timing_review["native_timing_qc_status"].value_counts(dropna=False).to_dict())
print("-" * 88)

display(
    fresh_timing_review[
        [
            "recording_id",
            "duration_status",
            "audio_hr_bpm",
            "audio_autocorr_peak",
            "native_hr_bpm",
            "native_autocorr_peak",
            "window_hr_iqr_bpm",
            "subset_disagreement_bpm",
            "hr_agreement_status",
            "native_specific_candidate",
            "native_timing_qc_status",
        ]
    ]
)

Fresh timing QC disagreement review
----------------------------------------------------------------------------------------
HR agreement counts:
{'disagree_gt_5bpm': 6, 'agree_5bpm': 3}
Native timing QC status counts:
{'caution_weak_native_autocorr;unstable_window_hr;subset_disagreement': 2, 'native_timing_supported': 1, 'caution_native_hr_upper_boundary;weak_native_autocorr': 1, 'caution_short_duration;native_hr_upper_boundary;weak_native_autocorr;subset_disagreement': 1, 'caution_native_hr_upper_boundary;weak_native_autocorr;unstable_window_hr;subset_disagreement': 1, 'caution_unstable_window_hr;subset_disagreement': 1, 'caution_subset_disagreement': 1, 'caution_weak_native_autocorr;subset_disagreement': 1}
----------------------------------------------------------------------------------------


,recording_id,duration_status,audio_hr_bpm,audio_autocorr_peak,native_hr_bpm,native_autocorr_peak,window_hr_iqr_bpm,subset_disagreement_bpm,hr_agreement_status,native_specific_candidate,native_timing_qc_status
0,202607090207400001SMP,duration_ok,78.9,0.157,78.9,0.444,2.0,0.5,agree_5bpm,True,native_timing_supported
1,202607090212220002SMP,duration_ok,95.2,0.707,200.0,0.168,7.1,0.0,disagree_gt_5bpm,True,caution_native_hr_upper_boundary;weak_native_a...
2,202607090213000003SMP,short_duration_caution,101.7,0.213,200.0,0.146,NaN,67.7,disagree_gt_5bpm,False,caution_short_duration;native_hr_upper_boundar...
3,202607090216270004SMP,duration_ok,92.3,0.627,200.0,0.154,149.2,16.2,disagree_gt_5bpm,False,caution_native_hr_upper_boundary;weak_native_a...
4,202607090217380005SMP,duration_ok,93.8,0.199,93.8,0.261,45.1,23.3,agree_5bpm,False,caution_unstable_window_hr;subset_disagreement
5,202607090223110006SMP,duration_ok,115.4,0.651,122.4,0.064,95.6,43.3,disagree_gt_5bpm,False,caution_weak_native_autocorr;unstable_window_h...
6,202607090226120007SMP,duration_ok,111.1,0.430,56.6,0.460,NaN,56.9,disagree_gt_5bpm,False,caution_subset_disagreement
7,202607090229140008SMP,duration_ok,78.9,0.675,153.8,0.188,56.8,25.0,disagree_gt_5bpm,False,caution_weak_native_autocorr;unstable_window_h...
8,202607090230590009SMP,duration_ok,75.0,0.502,76.9,0.140,NaN,48.7,agree_5bpm,False,caution_weak_native_autocorr;subset_disagreement


## Fresh batch modularization readiness checkpoint

### What this tests

This section summarizes whether the fresh July 9 native and AVI batch supports modularizing the NB13 helper functions.

### Why this matters

The fresh batch tests whether the helpers generalize beyond the original validation batch. It also identifies which outputs should remain caution-labeled rather than becoming hard defaults.


In [22]:
fresh_readiness_rows = [
    {
        "helper_group": "fresh native/AVI manifest",
        "planned_module": "paths / validation",
        "fresh_evidence": "9 recording folders; all have PW file, DcmRegionPara, and manually paired linked AVI",
        "fresh_result": "passed",
        "modularization_readiness": "ready as validation helper",
    },
    {
        "helper_group": "DcmRegionPara parser",
        "planned_module": "native_metadata",
        "fresh_evidence": "9/9 parsed; ROI/baseline/scale/time metadata consistent",
        "fresh_result": "passed",
        "modularization_readiness": "ready as stable helper",
    },
    {
        "helper_group": "native PW page loader",
        "planned_module": "native_qc",
        "fresh_evidence": "9/9 PW files divide cleanly into 1296-byte pages",
        "fresh_result": "passed",
        "modularization_readiness": "ready as stable low-level helper",
    },
    {
        "helper_group": "native counter summary",
        "planned_module": "native_qc",
        "fresh_evidence": "9/9 counter group interval = 25 pages; no resets; monotone fraction = 1.0",
        "fresh_result": "passed",
        "modularization_readiness": "ready as stable low-level helper",
    },
    {
        "helper_group": "hp_activity sidecar extraction",
        "planned_module": "native_qc",
        "fresh_evidence": "9/9 finite sidecar outputs; mixed-quality recordings produced caution labels",
        "fresh_result": "passed_with_cautions",
        "modularization_readiness": "ready as experimental additive QC helper",
    },
    {
        "helper_group": "native-specific Boolean rule",
        "planned_module": "native_qc",
        "fresh_evidence": "one recording was native_specific_candidate=True but hit 200 bpm boundary and disagreed with audio",
        "fresh_result": "rule_needs_revision",
        "modularization_readiness": "do not export as default acceptance rule",
    },
    {
        "helper_group": "AVI readability",
        "planned_module": "video",
        "fresh_evidence": "9/9 manually paired AVIs opened; first frames readable; 30 fps; 720x540",
        "fresh_result": "passed",
        "modularization_readiness": "ready as stable helper",
    },
    {
        "helper_group": "audio extraction and clipping QC",
        "planned_module": "audio_qc",
        "fresh_evidence": "9/9 audio extracted; 96 kHz; no clipping positives",
        "fresh_result": "passed",
        "modularization_readiness": "ready as additive QC helper",
    },
    {
        "helper_group": "audio/native HR agreement",
        "planned_module": "native_qc / audio_qc",
        "fresh_evidence": "3 agree_5bpm; 6 disagree_gt_5bpm; disagreements explained by caution labels in several cases",
        "fresh_result": "passed_as_qc_stress_test",
        "modularization_readiness": "export comparison helper with caution labels",
    },
]

fresh_modularization_readiness = pd.DataFrame(fresh_readiness_rows)

print("Fresh batch modularization readiness checkpoint")
print("-" * 88)
print(f"readiness rows:                    {len(fresh_modularization_readiness)}")
print("fresh result counts:")
print(fresh_modularization_readiness["fresh_result"].value_counts().to_dict())
print("-" * 88)

display(fresh_modularization_readiness)

Fresh batch modularization readiness checkpoint
----------------------------------------------------------------------------------------
readiness rows:                    9
fresh result counts:
{'passed': 6, 'passed_with_cautions': 1, 'rule_needs_revision': 1, 'passed_as_qc_stress_test': 1}
----------------------------------------------------------------------------------------


,helper_group,planned_module,fresh_evidence,fresh_result,modularization_readiness
0,fresh native/AVI manifest,paths / validation,"9 recording folders; all have PW file, DcmRegi...",passed,ready as validation helper
1,DcmRegionPara parser,native_metadata,9/9 parsed; ROI/baseline/scale/time metadata c...,passed,ready as stable helper
2,native PW page loader,native_qc,9/9 PW files divide cleanly into 1296-byte pages,passed,ready as stable low-level helper
3,native counter summary,native_qc,9/9 counter group interval = 25 pages; no rese...,passed,ready as stable low-level helper
4,hp_activity sidecar extraction,native_qc,9/9 finite sidecar outputs; mixed-quality reco...,passed_with_cautions,ready as experimental additive QC helper
5,native-specific Boolean rule,native_qc,one recording was native_specific_candidate=Tr...,rule_needs_revision,do not export as default acceptance rule
6,AVI readability,video,9/9 manually paired AVIs opened; first frames ...,passed,ready as stable helper
7,audio extraction and clipping QC,audio_qc,9/9 audio extracted; 96 kHz; no clipping posit...,passed,ready as additive QC helper
8,audio/native HR agreement,native_qc / audio_qc,3 agree_5bpm; 6 disagree_gt_5bpm; disagreement...,passed_as_qc_stress_test,export comparison helper with caution labels


## Final NB14 checkpoint

**Result classification:** fresh-batch modularization readiness passed with one rule revision.

The fresh July 9 batch intentionally includes short and poor-quality recordings to test pipeline robustness. The batch passed structural compatibility checks: all recordings contain native PW files, native `DcmRegionPara.txt`, and manually paired AVI files. The native PW files preserve the `1296` byte page structure, the `500 Hz` page-rate assumption, and the 20 Hz counter grouping seen in NB13.

Fresh `DcmRegionPara` parsing matched the NB13 metadata pattern across all recordings. Manually paired AVI files opened successfully, first frames were readable, and audio extraction succeeded for all recordings.

Native `hp_activity` sidecar extraction produced finite outputs for all recordings. The mixed-quality batch exposed a useful design requirement: native timing/QC should return caution labels rather than a single default acceptance Boolean. In particular, HR-like estimates that hit the upper search boundary or have weak autocorrelation support should be flagged.

The result supports modularizing stable low-level helpers for paths, video reading, native metadata parsing, native PW page loading, native counter summaries, audio clipping QC, and experimental native timing/QC sidecar extraction. It does not support exporting the old native-specific Boolean rule as a default acceptance rule.

This notebook does not claim clinical validation, native spectrogram recovery, native velocity-envelope recovery, native PSV, EDV, RI, PI, VTI, or diagnostic output.